# Model 2: Pretrained Transformer MCQ Solver

**Project:** Smart MCQ Solver · DL & GenAI · Milestone 3  
**Kernel path:** `nb/pretrained/pre_trained.ipynb`

---

## Architecture

Uses `bert-base-uncased` with HuggingFace `AutoModelForMultipleChoice` — all 5 options are scored in a single forward pass.

| Step | Description |
|------|-------------|
| **Tokenization** | Each question paired with each option: `[CLS] prompt [SEP] option [SEP]` |
| **Input shape** | `(batch, 5, seq_len)` — options stacked along dim-1 |
| **Encoder** | `bert-base-uncased` — 12 layers, 768 hidden dim |
| **Classifier** | Linear head over `[CLS]` → 5 logits |
| **Loss** | `CrossEntropyLoss` — ground truth is the correct option index |
| **Inference** | `torch.topk(logits, 3)` → top-3 letters → `submission.csv` |

## 1. Setup & Environment

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    get_linear_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import wandb
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
print('Imports ready.')

In [ ]:
# DeBERTa-v3-small — download from HuggingFace Hub on first run
MODEL_NAME = 'microsoft/deberta-v3-small'
print(f'Backbone : {MODEL_NAME}')


In [ ]:
KAGGLE_INPUT = '/kaggle/input/competitions/smart-mcq-solver-challenge'

def get_path(filename: str) -> str:
    # Checks local dev paths first, falls back to Kaggle mount
    candidates = [
        os.path.join('..', '..', 'data', filename),
        os.path.join('data', filename),
        os.path.join(KAGGLE_INPUT, filename),
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    return filename

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED   = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
print(f'Device : {DEVICE}')

## 2. Configuration & W&B Logging

In [ ]:
# W&B key read from environment — set WANDB_API_KEY in Kaggle secrets
_wandb_key = os.environ.get(
    'WANDB_API_KEY',
    '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'
)
wandb.login(key=_wandb_key, relogin=True)

run = wandb.init(
    project='23f2004343-t22026',
    name='Model_2_DeBERTa_v3_Merged',
    config={
        'base_model'     : 'microsoft/deberta-v3-small',
        'max_seq_length' : 256,
        'batch_size'     : 8,
        'epochs'         : 3,
        'learning_rate'  : 2e-5,
        'warmup_ratio'   : 0.1,
        'weight_decay'   : 0.01,
        'optimizer'      : 'AdamW',
        'loss_fn'        : 'CrossEntropyLoss',
        'num_options'    : 5,
    },
)
cfg = wandb.config
print(f'W&B run : {run.name}  |  project : {run.project}')
print(f'Config  : {dict(cfg)}')


## 3. Dataset & Tokenization

In [ ]:
trn_df = pd.read_csv(get_path('train.csv'))
tst_df = pd.read_csv(get_path('test.csv'))
print(f'train : {trn_df.shape}  |  test : {tst_df.shape}')
print(f'columns : {list(trn_df.columns)}')

OPTIONS = ['A', 'B', 'C', 'D', 'E']

# Load local tokenizer
# MODEL_NAME already set above; load tokenizer directly from hub
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LEN    = int(cfg.max_seq_length)


class MCQMultiChoiceDataset(Dataset):
    """Returns 5 tokenized (prompt, option) pairs per question.

    Tensors per item:
      input_ids      : (5, max_len)
      attention_mask : (5, max_len)
      label          : int in [0, 4] — correct option index (train only)
    """

    def __init__(self, df: pd.DataFrame, is_train: bool = True):
        self.df       = df.reset_index(drop=True)
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        prompt = str(row['prompt'])

        # Tokenize each (prompt, option) pair — output shape: (5, max_len)
        encodings = tokenizer(
            [prompt] * 5,
            [str(row[opt]) for opt in OPTIONS],
            max_length=MAX_LEN,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        item = {
            'input_ids'      : encodings['input_ids'],
            'attention_mask' : encodings['attention_mask'],
        }

        if self.is_train:
            ans           = str(row['answer']).strip().upper()
            label         = OPTIONS.index(ans) if ans in OPTIONS else 0
            item['label'] = torch.tensor(label, dtype=torch.long)

        return item


print('Dataset class defined.')

In [ ]:
trn_sub, val_sub = train_test_split(trn_df, test_size=0.1, random_state=SEED)

BATCH = int(cfg.batch_size)

trn_loader = DataLoader(
    MCQMultiChoiceDataset(trn_sub, is_train=True),
    batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True,
)
val_loader = DataLoader(
    MCQMultiChoiceDataset(val_sub, is_train=True),
    batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True,
)
tst_loader = DataLoader(
    MCQMultiChoiceDataset(tst_df, is_train=False),
    batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True,
)

probe = next(iter(trn_loader))
print(f"input_ids shape : {probe['input_ids'].shape}   # (batch, 5, max_len)")
print(f"labels          : {probe['label'].tolist()[:8]}")

## 4. Model & Evaluation Metrics

In [ ]:
# AutoModelForMultipleChoice: takes (batch, 5, seq_len), returns logits (batch, 5)
model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)
model.to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model  : {cfg.base_model}  |  trainable params : {n_params:,}')

criterion = nn.CrossEntropyLoss()
optimizer = AdamW(
    model.parameters(),
    lr=float(cfg.learning_rate),
    weight_decay=float(cfg.weight_decay),
    eps=1e-8,
)

total_steps  = len(trn_loader) * int(cfg.epochs)
warmup_steps = int(total_steps * float(cfg.warmup_ratio))
scheduler    = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)
print(f'Steps : {total_steps}  |  warmup : {warmup_steps}')

In [ ]:
def ap_at_3(ranked: list, correct: int) -> float:
    """Average Precision at 3 for one question.

    Viva Defence: AP@3 = 1/rank if the correct answer appears in top-3,
    else 0. Averaging over all questions gives MAP@3 — the official metric.
    """
    hits, score = 0, 0.0
    for k, pred in enumerate(ranked[:3], start=1):
        if pred == correct:
            hits  += 1
            score += hits / k
    return score


def run_eval(model, loader):
    """Validation pass — returns loss, accuracy, macro-F1, and MAP@3.

    MAP@3 is computed passively from raw logits via argsort; it does not
    influence checkpoint selection or gradient flow.
    """
    model.eval()
    total_loss, all_preds, all_labels, all_ap3 = 0.0, [], [], []

    with torch.no_grad():
        for batch in loader:
            ids   = batch['input_ids'].to(DEVICE)
            masks = batch['attention_mask'].to(DEVICE)
            labs  = batch['label'].to(DEVICE)

            out    = model(input_ids=ids, attention_mask=masks)
            logits = out.logits  # (batch, 5) — raw un-softmaxed scores
            loss   = criterion(logits, labs)
            total_loss += loss.item()

            # Rank all 5 options descending by logit; no softmax needed for ranking
            ranked_idx = torch.argsort(logits, dim=1, descending=True)[:, :3].cpu().tolist()
            preds      = [r[0] for r in ranked_idx]  # top-1 for accuracy/F1
            labels     = labs.cpu().tolist()

            all_preds.extend(preds)
            all_labels.extend(labels)

            # Passive MAP@3 — purely for W&B logging, zero effect on training
            for ranked, correct in zip(ranked_idx, labels):
                all_ap3.append(ap_at_3(ranked, correct))

    n = max(len(loader), 1)
    return {
        'val_loss'     : total_loss / n,
        'val_accuracy' : accuracy_score(all_labels, all_preds),
        'val_f1_macro' : f1_score(all_labels, all_preds, average='macro', zero_division=0),
        'val_map3'     : float(np.mean(all_ap3)) if all_ap3 else 0.0,
    }


## 5. Training Loop

In [ ]:
best_val_loss = float('inf')
best_val_map3 = 0.0
EPOCHS        = int(cfg.epochs)

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0

    for batch in trn_loader:
        ids   = batch['input_ids'].to(DEVICE)
        masks = batch['attention_mask'].to(DEVICE)
        labs  = batch['label'].to(DEVICE)

        optimizer.zero_grad()
        logits = model(input_ids=ids, attention_mask=masks).logits
        loss   = criterion(logits, labs)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # prevent exploding gradients
        optimizer.step()
        scheduler.step()
        running_loss += loss.item()

    avg_trn = running_loss / max(len(trn_loader), 1)
    val_m   = run_eval(model, val_loader)
    best_val_map3 = max(best_val_map3, val_m['val_map3'])

    wandb.log({'epoch': epoch, 'train_loss': avg_trn, **val_m})
    print(
        f'Epoch {epoch}/{EPOCHS}  '
        f'trn_loss={avg_trn:.4f}  '
        f'val_loss={val_m["val_loss"]:.4f}  '
        f'val_acc={val_m["val_accuracy"]:.4f}  '
        f'val_f1={val_m["val_f1_macro"]:.4f}  '
        f'val_map3={val_m["val_map3"]:.4f}'
        f'val_map3={val_m["val_map3"]:.4f}'
    )

    # Save best checkpoint
    if val_m['val_loss'] < best_val_loss:
        best_val_loss = val_m['val_loss']
        torch.save(model.state_dict(), '/kaggle/working/deberta_mcq_best.pt')
        print(f'  ✓ Checkpoint saved (val_loss={best_val_loss:.4f})')

wandb.run.summary['best_val_loss'] = best_val_loss
# Checkpoint saved on val_loss — MAP@3 is logged passively; does not drive checkpoint
wandb.run.summary['best_val_map3'] = best_val_map3
print('\nTraining complete.')

## 6. Inference & Submission

In [ ]:
# Load best checkpoint
ckpt = '/kaggle/working/deberta_mcq_best.pt'
if os.path.exists(ckpt):
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    print(f'Loaded best checkpoint: {ckpt}')

model.eval()
OPTIONS_ARR = np.array(OPTIONS)
all_top3    = []

print(f'Running batched inference on {len(tst_df)} test questions...')
with torch.no_grad():
    for batch in tst_loader:
        ids   = batch['input_ids'].to(DEVICE)
        masks = batch['attention_mask'].to(DEVICE)

        logits   = model(input_ids=ids, attention_mask=masks).logits  # (B, 5)
        top3_idx = torch.topk(logits, 3, dim=1).indices.cpu().numpy()  # (B, 3) descending

        for row_idx in top3_idx:
            all_top3.append(' '.join(OPTIONS_ARR[row_idx]))

# Align column names with sample_submission.csv
sub_template = pd.read_csv(get_path('sample_submission.csv'))
print(f'sample_submission columns : {list(sub_template.columns)}')
print(f'Predictions generated     : {len(all_top3)}')

sub_df = pd.DataFrame({
    sub_template.columns[0]: tst_df['id'].values,
    sub_template.columns[1]: all_top3,
})

sub_df.to_csv('submission.csv', index=False)
print(f'submission.csv written — {len(sub_df)} rows')
print(sub_df.head())

# Log artifact and close run
art = wandb.Artifact('submission_deberta_mcq', type='predictions')
art.add_file('submission.csv')
wandb.log_artifact(art)
wandb.finish()
print('W&B run closed.')